# 10.9M Hindi–Sanskrit decoder

This is the first trained size in experiment `010`. The trainable count is **10,942,720**. The command flag is `--model 10m`. The blueprint is `configs/model_10m.yaml`.

The sibling notebooks are `model_18m.ipynb` and `model_50m.ipynb`. Run the code cells from `experiments/language/010-hindi-sanskrit-slm`. They build the network and, when a checkpoint is on disk, print greedy translations. They do not start training.


## Architecture

Grouped-query attention, SwiGLU, RMSNorm, and rotary positions with base 100,000. The input embedding and the output head are one matrix.

| Piece | Setting |
| --- | --- |
| Vocabulary | 12,000 |
| Width | 256 |
| Layers | 12 |
| Query heads | 8 |
| Key-value heads | 2 |
| Head dimension | 32 |
| SwiGLU intermediate | 640 |
| Context | 1,024 |
| Dropout | 0.05 |
| Tied embeddings | yes |

| Weights | Count |
| --- | --- |
| Tied token embedding, 12,000 × 256 | 3,072,000 |
| One block: attention 163,840 + SwiGLU 491,520 + two RMSNorm scales 512 | 655,872 |
| Twelve blocks | 7,870,464 |
| Final RMSNorm | 256 |
| **Total** | **10,942,720** |

Rotary tables are buffers. Output projections start at `0.02 / sqrt(2 * 12)`.


In [1]:
from pathlib import Path
import sys
import yaml

HERE = Path.cwd()
ROOT = HERE if (HERE / "configs" / "model_10m.yaml").exists() else HERE / "experiments" / "language" / "010-hindi-sanskrit-slm"
sys.path.insert(0, str(ROOT))
print("experiment", ROOT)

CONFIG_NAME = "configs/model_10m.yaml"
EXPECTED = 10_942_720

from src.model import build_model, count_parameters

config = yaml.safe_load((ROOT / CONFIG_NAME).read_text())
counted = count_parameters(config)
model = build_model(config)
live = model.num_parameters()
print("config", CONFIG_NAME)
print("formula", f"{counted:,}")
print("built", f"{live:,}")
print("expected", f"{EXPECTED:,}")
print("tied", model.lm_head.weight.data_ptr() == model.wte.weight.data_ptr())
print(
    "layers", len(model.blocks),
    "width", config["n_embd"],
    "query heads", config["n_head"],
    "kv heads", config["n_kv_head"],
    "head dim", config["head_dim"],
    "context", config["block_size"],
    "vocab", config["vocab_size"],
)
assert live == EXPECTED == counted


experiment /Users/akashchauhan/Development/ai-models-learning/experiments/language/010-hindi-sanskrit-slm
config configs/model_10m.yaml
formula 10,942,720
built 10,942,720
expected 10,942,720
tied True
layers 12 width 256 query heads 8 kv heads 2 head dim 32 context 1024 vocab 12000


## Data and tokenizer

Tokenizer: `tokenizers/sp.model`, SentencePiece Unigram, 12,000 pieces, identity normalization, no byte fallback.

This size was pretrained on `data/clean/pretrain.jsonl` and fine-tuned on `data/clean/sft.jsonl`. That cleaned mix includes Hindi and Sanskrit Wikipedia plus Samasāmayik and the English-pivoted BPCC pairs. Later sizes kept Wikipedia out. The official Samasāmayik test stayed held out.

Pretraining uses `<hi>` and `<sa>` and stops at a token budget. Fine-tuning loads those weights into a new AdamW and learns `<translate_hi_sa>`, `<translate_sa_hi>`, and the other task tags. Loss is next-token cross-entropy. Fine-tuning masks every token through `<tgt>`.


## What was trained

| Stage | Stop | Checkpoint |
| --- | --- | --- |
| Pretrain | 100M tokens, then continued to 200M | `runs/pretrain/tok-100m`, `runs/pretrain/tok-200m` |
| Supervised fine-tune | about 20M supervised tokens from each boundary | `runs/sft/from-tok-100m`, `runs/sft/from-tok-200m` |

Learning rate: pretrain peak 6e-4, floor 6e-5, warmup 2% of the first budget. Fine-tune peak 2e-4, floor 2e-5.

On the mixed internal validation split, the fine-tune from the 200M checkpoint scored cross-entropy 3.36 and chrF **26.57**, against a copy baseline of 25.63.

Weights and the reproduce bundle are in `gs://niramana-ai-model-weights-dev/hindi-sanskrit-10m/`. Checkpoints are not in git.


In [1]:
CHECKPOINTS = ["runs/sft/from-tok-200m"]

from src.sample import continue_prompt, load_checkpoint

PROMPTS = [
    ("translate_hi_sa", "पानी ठंडा है।"),
    ("translate_hi_sa", "मैं किताब पढ़ता हूँ।"),
    ("translate_sa_hi", "जलं शीतलम् अस्ति।"),
]

def show(directory, task, prompt, max_new_tokens=32):
    path = ROOT / directory
    if not (path / "checkpoint.pt").exists():
        print(f"missing {directory}")
        return None
    text, checkpoint = continue_prompt(
        path, task, prompt, max_new_tokens=max_new_tokens, temperature=0.0,
        model=show.model, tokenizer=show.tokenizer, device=show.device,
    )
    body = text.split("<tgt>", 1)[-1].strip() if "<tgt>" in text else text.strip()
    print(f"{task}")
    print(f"  prompt: {prompt}")
    print(f"  output: {body}")
    return checkpoint

for directory in CHECKPOINTS:
    path = ROOT / directory
    print("\n#", directory)
    if not (path / "checkpoint.pt").exists():
        print("checkpoint is not on this machine")
        continue
    show.model, show.tokenizer, checkpoint, show.device = load_checkpoint(path)
    print("stage", checkpoint["stage"], "tokens", f"{checkpoint['tokens_seen']:,}")
    for task, prompt in PROMPTS:
        show(directory, task, prompt)



# runs/sft/from-tok-200m
stage sft tokens 20,000,016
translate_hi_sa
  prompt: पानी ठंडा है।
  output: जलं लम्ब: अस्ति।
translate_hi_sa
  prompt: मैं किताब पढ़ता हूँ।
  output: अहं पुस्तकं पठामि ।
translate_sa_hi
  prompt: जलं शीतलम् अस्ति।
  output: जल को बंद करना है।


## Commands

```bash
uv run python -m src.smoke --model 10m
uv run python -m src.train --stage pretrain --model 10m --until-tokens 100000000
uv run python -m src.train --stage sft --model 10m --init runs/pretrain/tok-200m
uv run python -m src.sample --checkpoint runs/sft/from-tok-200m --task translate_hi_sa --prompt "पानी ठंडा है।"
```
